In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
## Load the ANN Trained model,scaler pickle,onehot
model = load_model('model.h5')

#load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open("scaler.pkl",'rb') as file:
    scaler = pickle.load(file)

EOFError: Ran out of input

In [ ]:
#Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [ ]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [ ]:
## Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [ ]:
# One-hot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\chauh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [ ]:
## concatanation with one hot encoded data 
input_df = pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0



# Difference Between fit(), transform(), and fit_transform()

## 1. fit()
- Learns the parameters from the given data.
- Does not transform the data.

**Example:**

Suppose Age = [20, 30, 40, 50, 60]

Using `fit()`, the scaler learns:
- Minimum = 20
- Maximum = 60

## 2. transform()
- Applies the already learned parameters to transform the data.
- Does not learn anything new.

**Example:**

Using the previously learned Minimum = 20 and Maximum = 60:

| Original Value | Transformed Value |
|---|---|
| 20 | 0.00 |
| 30 | 0.25 |
| 40 | 0.50 |
| 50 | 0.75 |
| 60 | 1.00 |

## 3. fit_transform()
- Combines both fit() and transform().
- First learns the parameters and then transforms the data.

**Example:**

Original Age = [20, 30, 40, 50, 60]

1. Learns Minimum = 20 and Maximum = 60.
2. Converts all values into the range 0 to 1.

---

## 4. How to Decide Whether to Use fit() or transform()?

Ask yourself: **Have I already learned the parameters from this data?**

### Case 1: Training Data
- When working with training data for the first time, use `fit_transform()`.
- It learns the parameters and transforms the data.

**Example:**

Training Age = [20, 30, 40, 50, 60]

The scaler learns Minimum = 20 and Maximum = 60, then scales the values.

### Case 2: Testing Data
- When working with testing data, use `transform()`.
- The scaler already knows the parameters from training data.
- It simply applies those same parameters to the testing data.

**Example:**

Testing Age = [25, 35, 45]

The scaler uses the previously learned Minimum = 20 and Maximum = 60.

### Case 3: New / Production Data
- When new data arrives after model training, use `transform()`.
- Reuse the parameters learned during training.

---

## 5. Comparison

| Method | Meaning | When to Use |
|---|---|---|
| fit() | Learn parameters | When learning parameters separately |
| transform() | Apply learned parameters | Testing / New Data |
| fit_transform() | Learn + Apply | Training Data |

## 6. Important: Data Leakage
- Use `fit_transform()` on training data.
- Use `transform()` on testing data.
- Never fit separately on testing data because it causes Data Leakage.
- This ensures that information from testing data does not influence the training process.

## 7. Remember

- fit() = Learn
- transform() = Apply
- fit_transform() = Learn + Apply

**Golden Rule:**
Training Data → fit_transform()

Testing Data → transform()

New Production Data → transform()


In [ ]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [ ]:
## Predict churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


array([[0.03554427]], dtype=float32)

In [ ]:
prediction_proba = prediction[0][0]

In [ ]:
prediction_proba

np.float32(0.035544265)

In [ ]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
